# TDT-FL — 02. Protocole de données (eICU-CRD) — version 1.0

Ce notebook construit le protocole expérimental de TDT-FL **à partir des sorties du notebook 01 d'OP-FL** (`data_v1/`). Il ne relit aucune table eICU et **n'écrit rien dans `data_v1/`** : toutes ses sorties vont dans `data_tdtfl/`.
Chaque étape tient dans une seule cellule ; exécutez-les dans l'ordre.

| Étape | Contenu | Sorties principales |
|---|---|---|
| 0 | Configuration, chemins, outils communs | — |
| 1 | Hôpitaux : éligibilité (≥ 20 décès), strates, hôpitaux de réglage, régime temporel, plis R3 | (écrit à l'étape 7) |
| 2 | Phénotypes (diagnostic d'admission APACHE) et axe opératoire | (écrit à l'étape 7) |
| 3 | Variables tabulaires des 24 h et disponibilité des variables par hôpital | (écrit à l'étape 7) |
| 4 | Découpages R1 (statique), 10 graines | (écrit à l'étape 7) |
| 5 | Découpages R2 (temporel 2014 → 2015) | (écrit à l'étape 7) |
| 6 | Réservoirs d'étiquetage R3 (hôpital non vu) | (écrit à l'étape 7) |
| 7 | Protocole figé, vérifié par hash, puis écriture des fichiers | `protocol_tdtfl.json`, `splits.parquet`, `stays.parquet`, `hospitals.csv`, `phenotype_map.csv`, `features_raw.parquet`, `feature_meta.csv`, `availability.csv` |
| 8 | Module de chargement pour les notebooks suivants | `src/tdtfl_data.py` |
| 9 | Contrôles de cohérence | — |
| 10 | Effectifs utiles aux expériences et fiche de données | `reports/`, `DATA_CARD_TDTFL.md` |

**Repris d'OP-FL sans changement** : la cohorte (critères patients, exclusion des hôpitaux sans labo ni monitorage), les labels (mortalité, AKI) et les tenseurs bruts par hôpital (`clients/hospital_<id>.npz`).

**Propre à TDT-FL**
* Un protocole séparé. Le `splits/protocol.json` d'OP-FL (hash `d92099ea7c11d517`) n'est ni utilisé ni modifié.
* Les hôpitaux de 100 à 499 séjours, mis à l'écart dans OP-FL pour le test verrouillé, **participent comme clients** dès qu'ils ont au moins 20 décès : ce sont les cibles où le transfert a le plus de marge.
* Des variables tabulaires (résumés des 24 premières heures), et non des séquences : le prédicteur n'est pas la contribution, et les petits hôpitaux n'ont pas assez de données pour un modèle séquentiel.
* Une normalisation globale, recalculée au chargement pour chaque régime et chaque graine, et une harmonisation des variables indisponibles (étape 3).

**Principes**
1. D ne sert qu'à décider (choisir et autoriser un transfert) ; le test ne sert qu'à mesurer.
2. Aucune statistique utilisée en entrée d'un modèle n'est calculée sur D ou sur le test.
3. Les labels n'interviennent que dans le seuil de 20 décès par hôpital et dans la stratification des découpages (décès × phénotype). Les deux s'appliquent de la même façon à toutes les méthodes comparées.
4. Le protocole est tiré une seule fois, puis vérifié à chaque exécution : il ne peut pas être re-tiré par accident.

## Étape 0 — Configuration et outils communs

Adaptez uniquement les chemins si votre arborescence diffère (ce notebook est supposé dans `notebooks/`, à côté de `data_v1/`).

In [12]:
# ============================================================
# ÉTAPE 0 — CONFIGURATION, CHEMINS ET OUTILS COMMUNS
# ============================================================
import json, hashlib, sys, importlib, re, warnings
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 80)

# ---------- Chemins (à adapter) ----------
PROJECT_DIR = Path("../..")
OPFL_DIR = PROJECT_DIR / "data_v1"       # sorties du notebook 01 d'OP-FL (LECTURE SEULE)
TDT_DIR = PROJECT_DIR / "data_tdtfl"     # sorties de ce notebook
SRC_DIR = PROJECT_DIR / "src"
REPORTS = TDT_DIR / "reports"

REQUIRED = [OPFL_DIR / "interim" / "cohort_final.parquet", OPFL_DIR / "interim" / "outcomes.parquet",
            OPFL_DIR / "features.json", OPFL_DIR / "clients" / "manifest.csv"]
missing = [str(p) for p in REQUIRED if not p.exists()]
if missing:
    raise FileNotFoundError(f"Sorties du notebook 01 introuvables : {missing}")
if TDT_DIR.resolve() == OPFL_DIR.resolve() or OPFL_DIR.resolve() in TDT_DIR.resolve().parents:
    raise ValueError("TDT_DIR doit être en dehors de data_v1/ : ce notebook n'écrit rien dans les sorties d'OP-FL.")
for d in (TDT_DIR, REPORTS, SRC_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Empreinte du protocole OP-FL : on vérifiera à la fin qu'il n'a pas été touché.
OPFL_PROTOCOL = OPFL_DIR / "splits" / "protocol.json"
OPFL_PROTOCOL_SHA = hashlib.sha256(OPFL_PROTOCOL.read_bytes()).hexdigest() if OPFL_PROTOCOL.exists() else None

# ---------- Protocole TDT-FL ----------
PROTOCOL_VERSION = "1.1"
BASE_SEED = 2026                 # graine du protocole (indépendante de celle d'OP-FL)
N_SEEDS = 10                     # graines de découpage patients
MIN_DEATHS = 20                  # hôpital éligible si >= 20 décès sur 2014-2015
STRATA = {"petit": (20, 59), "moyen": (60, 149), "grand": (150, None)}   # nombre de décès
TUNE_FRACTION = 0.20             # part des hôpitaux de chaque strate réservée au réglage des hyperparamètres
YEARS = (2014, 2015)
MIN_DEATHS_PER_YEAR = 10         # régime temporel : >= 10 décès chaque année
N_R3_FOLDS = 5                   # plis d'hôpitaux pour le régime « hôpital non vu »
R3_BUDGETS = (50, 100, 200)      # patients étiquetés disponibles chez l'hôpital non vu
N_INNER_FOLDS = 5                # plis internes du train (régularisation, calibration hors échantillon)
AVAIL_MIN_COVERAGE = 0.05        # variable disponible dans un hôpital si >= 5 % des séjours l'observent
N_TOP_DX = 20                    # diagnostics d'admission codés en one-hot
MIN_UNIT_SHARE = 0.01            # type d'unité codé s'il couvre >= 1 % des séjours de réglage
SUBGROUP_DEATH_THRESHOLDS = (5, 10)   # seuils rapportés pour la faisabilité de E2

# ---------- Codes des découpages ----------
R1_CODES = {"train": 0, "D": 1, "test": 2}
R2_CODES = {"train14": 0, "D14": 1, "test14": 2, "D15": 3, "test15": 4}
R3_POOLS = {"cal": 0, "D": 1}
R1_PATTERN = [0, 0, 0, 1, 2]     # blocs de 5 : 3 train, 1 D, 1 test  -> 60 / 20 / 20
R2_PATTERN_Y1 = [0, 0, 0, 1, 2]  # 2014 : train14 / D14 / test14     -> 60 / 20 / 20
R2_PATTERN_Y2 = [3, 4]           # 2015 : D15 / test15                -> 50 / 50
assert STRATA["petit"][0] == MIN_DEATHS

# ---------- Outils communs ----------
def save_json(obj, path):
    Path(path).write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def rng_for(*keys):
    """Générateur aléatoire déterminé par (BASE_SEED, *keys) : un tirage ne dépend pas de l'ordre des boucles."""
    return np.random.default_rng([BASE_SEED, *[int(k) for k in keys]])

def blocked_allocation(keys, pattern, rng):
    """Répartit n éléments selon `pattern` (liste de codes), stratifié sur `keys`.
    Ordre aléatoire dans chaque strate, strates mises bout à bout, puis blocs consécutifs de len(pattern)
    éléments qui reçoivent chacun une permutation aléatoire du motif. Chaque strate reçoit ainsi les
    proportions du motif à un ou deux éléments près, même quand elle est petite (décès d'un phénotype)."""
    keys = np.asarray(keys)
    pattern = np.asarray(pattern, dtype=np.int8)
    n, B = len(keys), len(pattern)
    order = rng.permutation(n)
    order = order[np.argsort(keys[order], kind="stable")]
    out = np.empty(n, dtype=np.int8)
    for start in range(0, n, B):
        blk = order[start:start + B]
        out[blk] = rng.permutation(pattern)[:len(blk)]
    return out

def interleaved_rank(y, rng):
    """Ordre d'acquisition : tout préfixe garde la proportion de chaque classe, à un élément près."""
    y = np.asarray(y)
    n = len(y)
    u = np.empty(n)
    for cls in np.unique(y):
        idx = np.flatnonzero(y == cls)
        u[idx] = (rng.permutation(len(idx)) + rng.random()) / len(idx)
    order = np.lexsort((rng.random(n), u))
    rank = np.empty(n, dtype=np.int32)
    rank[order] = np.arange(n)
    return rank

def stratum_of(n_deaths):
    for name, (lo, hi) in STRATA.items():
        if n_deaths >= lo and (hi is None or n_deaths <= hi):
            return name
    return "exclu"

def digest_df(df):
    text = df.to_csv(index=False, lineterminator="\n", float_format="%.6f")
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]

def md_table(df, index=True):
    """Tableau Markdown sans dépendance externe (pour la fiche de données)."""
    d = df.reset_index() if index else df
    head = "| " + " | ".join(map(str, d.columns)) + " |"
    sep = "|" + "---|" * len(d.columns)
    rows = ["| " + " | ".join("" if pd.isna(v) else str(v) for v in r) + " |" for r in d.itertuples(index=False)]
    return "\n".join([head, sep] + rows)

print("Entrées (lecture seule) :", OPFL_DIR.resolve())
print("Sorties                 :", TDT_DIR.resolve())
print("Protocole OP-FL         :", "présent, empreinte enregistrée" if OPFL_PROTOCOL_SHA else "absent")

Entrées (lecture seule) : C:\Users\Stryve\Desktop\StryveSolution\Imagerie-Phd\P.H.D\New Impl\FUALA-w_28-06-2026\OP-FL-V2\data_v1
Sorties                 : C:\Users\Stryve\Desktop\StryveSolution\Imagerie-Phd\P.H.D\New Impl\FUALA-w_28-06-2026\OP-FL-V2\data_tdtfl
Protocole OP-FL         : présent, empreinte enregistrée


## Étape 1 — Hôpitaux

| Règle | Définition |
|---|---|
| Éligibilité | au moins **20 décès** sur 2014–2015 (en dessous, D et le test d'un hôpital ne contiennent que 2 ou 3 décès) |
| Strates | par nombre de décès : **petits** 20–59, **moyens** 60–149, **grands** ≥ 150 |
| Hôpitaux de réglage (`tune`) | 20 % de chaque strate, tirés une fois pour toutes. Ils servent de **cibles** pour régler les hyperparamètres de décision (η, ρ, taille de lot, δ…) ; leurs résultats ne sont pas rapportés. Ils restent des **sources** pour tous les hôpitaux. |
| Hôpitaux d'évaluation (`eval`) | tous les autres : ce sont les cibles dont les résultats sont rapportés |
| Régime temporel R2 | au moins **10 décès en 2014 et 10 en 2015** |
| Plis R3 | 5 plis d'hôpitaux, équilibrés par strate. Quand une cible du pli k est « non vue », la fédération est formée des hôpitaux hors du pli k. |

Avec vos chiffres : 101 hôpitaux éligibles (50 petits, 40 moyens, 11 grands), dont 88 pour le régime temporel.

In [13]:
# ============================================================
# ÉTAPE 1 — HÔPITAUX : ÉLIGIBILITÉ, STRATES, RÔLES, RÉGIME TEMPOREL, PLIS R3
# ============================================================
cohort = pd.read_parquet(OPFL_DIR / "interim" / "cohort_final.parquet")
outcomes = pd.read_parquet(OPFL_DIR / "interim" / "outcomes.parquet")
assert cohort["patientunitstayid"].is_unique, "un séjour apparaît deux fois dans cohort_final"
assert cohort["uniquepid"].is_unique, "le protocole suppose un seul séjour par patient (option d'OP-FL)"

out_cols = [c for c in ["y_aki", "aki_ascertained", "los_icu_days", "y_los_gt7", "apache_pred_mortality"]
            if c in outcomes.columns]
stays_all = cohort.merge(outcomes[["patientunitstayid"] + out_cols], on="patientunitstayid",
                         how="left", validate="one_to_one")
if stays_all["hospitaldischargeyear"].isna().any():
    raise ValueError(f"{int(stays_all['hospitaldischargeyear'].isna().sum())} séjours sans année de sortie : "
                     "décidez de leur sort avant de tirer le protocole (régime temporel).")
stays_all["year"] = stays_all["hospitaldischargeyear"].astype(int)
assert set(stays_all["year"].unique()) <= set(YEARS), f"années inattendues : {sorted(stays_all['year'].unique())}"
stays_all = stays_all.rename(columns={"role": "opfl_role"})

# ---- effectifs par hôpital et par année ----
g = stays_all.groupby(["hospitalid", "year"])["y_mortality"]
hy = pd.concat({"n": g.size(), "deaths": g.sum()}, axis=1).unstack("year", fill_value=0)
hy.columns = [f"{a}_{b}" for a, b in hy.columns]
for col in [f"{a}_{y}" for a in ("n", "deaths") for y in YEARS]:
    if col not in hy.columns:
        hy[col] = 0
hy = hy[[f"{a}_{y}" for a in ("n", "deaths") for y in YEARS]].astype(int)

gh = stays_all.groupby("hospitalid")
hospitals = pd.DataFrame({"n_stays": gh.size(), "n_deaths": gh["y_mortality"].sum().astype(int),
                          "opfl_role": gh["opfl_role"].first()}).join(hy)
hospitals["mortality"] = (hospitals["n_deaths"] / hospitals["n_stays"]).round(4)
hospitals["eligible"] = hospitals["n_deaths"] >= MIN_DEATHS
hospitals["stratum"] = np.where(hospitals["eligible"], hospitals["n_deaths"].map(stratum_of), "exclu")
hospitals["temporal"] = (hospitals["eligible"]
                         & (hospitals[f"deaths_{YEARS[0]}"] >= MIN_DEATHS_PER_YEAR)
                         & (hospitals[f"deaths_{YEARS[1]}"] >= MIN_DEATHS_PER_YEAR))

# ---- rôles (réglage / évaluation) et plis R3 : tirés une seule fois, stratifiés par strate ----
rng = rng_for(0)
hospitals["role"] = np.where(hospitals["eligible"], "eval", "exclu")
hospitals["r3_fold"] = -1
counter = 0
for s in STRATA:
    ids = np.sort(hospitals.index[hospitals["stratum"] == s].to_numpy())
    n_tune = int(round(TUNE_FRACTION * len(ids)))
    hospitals.loc[rng.permutation(ids)[:n_tune], "role"] = "tune"
    for h in rng.permutation(ids):
        hospitals.loc[h, "r3_fold"] = counter % N_R3_FOLDS
        counter += 1
hospitals["r3_fold"] = hospitals["r3_fold"].astype(int)
ELIGIBLE = np.sort(hospitals.index[hospitals["eligible"]].to_numpy()).astype(int)

# ---- séjours retenus ----
stays = stays_all[stays_all["hospitalid"].isin(ELIGIBLE)].copy()
for c in ["stratum", "role", "temporal", "r3_fold"]:
    stays[c] = stays["hospitalid"].map(hospitals[c])
stays = stays.sort_values(["hospitalid", "patientunitstayid"]).reset_index(drop=True)

# ---- résumé ----
el = hospitals[hospitals["eligible"]]
summary = el.groupby("stratum").agg(hopitaux=("n_stays", "size"), sejours=("n_stays", "sum"),
                                    deces=("n_deaths", "sum"), deces_median=("n_deaths", "median"),
                                    reglage=("role", lambda r: int((r == "tune").sum())),
                                    evaluation=("role", lambda r: int((r == "eval").sum())),
                                    temporel=("temporal", "sum")).reindex(list(STRATA))
summary.loc["total"] = summary.sum(numeric_only=True)
summary.loc["total", "deces_median"] = el["n_deaths"].median()
summary = summary.astype({c: int for c in summary.columns if c != "deces_median"})
display(summary)
excl = hospitals[~hospitals["eligible"]]
print(f"Hôpitaux exclus (< {MIN_DEATHS} décès) : {len(excl)} hôpitaux, {excl['n_stays'].sum()} séjours, "
      f"{excl['n_deaths'].sum()} décès")
print(f"Séjours retenus : {len(stays)} ({stays['y_mortality'].sum()} décès, mortalité {stays['y_mortality'].mean():.2%})")
print("Rôle OP-FL des hôpitaux retenus :", el["opfl_role"].value_counts().to_dict())
print("Composition des plis R3 (hôpitaux par strate) :")
display(pd.crosstab(el["r3_fold"], el["stratum"])[list(STRATA)])

,hopitaux,sejours,deces,deces_median,reglage,evaluation,temporel
stratum,,,,,,,
petit,50,20595,1705,31.5,10,40,39
moyen,40,34900,3399,79.0,8,32,38
grand,11,24378,2511,217.0,2,9,11
total,101,79873,7615,60.0,20,81,88


Hôpitaux exclus (< 20 décès) : 55 hôpitaux, 10861 séjours, 606 décès
Séjours retenus : 79873 (7615 décès, mortalité 9.53%)
Rôle OP-FL des hôpitaux retenus : {'dev': 60, 'locked': 41}
Composition des plis R3 (hôpitaux par strate) :


stratum,petit,moyen,grand
r3_fold,,,
0,10,8,3
1,10,8,2
2,10,8,2
3,10,8,2
4,10,8,2


## Étape 2 — Phénotypes et axe opératoire

Un phénotype par séjour, tiré de `apacheadmissiondx`. La taxonomie suit les systèmes d'organes d'APACHE IV, avec le sepsis isolé en premier parce que c'est le premier motif d'admission et le plus létal. ClusterFL reposait sur des clients synthétiques : il n'y a donc pas de correspondance antérieure à respecter, et celle-ci est justifiée a priori.

| Priorité | Phénotype | Exemples de motifs (insensibles à la casse) |
|---|---|---|
| 1 | `sepsis` | sepsis, septic |
| 2 | `neurologique` | CVA, intracranial, subdural, seizure, coma, head trauma, cranial, laminectomy… |
| 3 | `cardiovasculaire` | cardiac, infarction, MI, angina, CHF, rhythm, CABG, valve, aortic, endarterectomy, aneurysm, hypertension… |
| 4 | `respiratoire` | pneumonia, respiratory, pulmonary, emphysema, asthma, ARDS, airway, pleural, thoracotomy, embolus… |
| 5 | `autre` | tout le reste (hémorragies digestives, métabolique, rénal, intoxications, chirurgie digestive, diagnostic manquant…) |

* Le texte entre parenthèses est retiré avant la recherche des motifs : « Cardiac arrest (… for respiratory arrest see Respiratory System) » ne doit pas devenir respiratoire, ni « Hypertension, uncontrolled (for cerebrovascular accident…) » neurologique.
* **Axe secondaire, opératoire** : `operative = 1` si `unitadmitsource` vaut Operating Room, Recovery Room ou PACU. Il sert à l'analyse de sensibilité de E2 (et la concordance avec un motif chirurgical du diagnostic est affichée).
* La correspondance complète, diagnostic par diagnostic, est écrite dans `phenotype_map.csv` (étape 7) : **relisez-la avant E1**. Toute correction passe par `PHENOTYPE_OVERRIDES`, et change le hash du protocole.

In [14]:
# ============================================================
# ÉTAPE 2 — PHÉNOTYPES (apacheadmissiondx) ET AXE OPÉRATOIRE
# ============================================================
PHENOTYPES = ["sepsis", "cardiovasculaire", "neurologique", "respiratoire", "autre"]
PHENOTYPE_RULES = [   # (phénotype, motif) — l'ordre est la priorité
    ("sepsis", r"\bsepsis\b|\bseptic\b"),
    ("neurologique", r"\bcva\b|cerebrovascular|stroke|intracranial|subarachnoid|subdural|epidural|seizure|\bcoma\b"
                     r"|level of consciousness|head only trauma|head trauma|head/|cranial|craniotomy|neurolog|laminectomy"
                     r"|spinal|encephal|mening|guillain|myasthenia|neuromuscular|ventriculostomy|hydroceph|brain"),
    ("cardiovasculaire", r"cardiac|myocardial|\bmi\b|infarction|angina|chest pain|\bchf\b|heart failure|rhythm|arrhythm"
                         r"|\bcabg\b|coronary|valve|aortic|mitral|tricuspid|endarterectomy|aneurysm|arterial|vascular"
                         r"|hypertension|pericard|tamponade|cardiomyopathy|endocarditis|(?<!non-)cardiogenic|cardiovascular"
                         r"|pacemaker|defibrillator|bypass|aorto|femoral|carotid|ventricular"),
    ("respiratoire", r"pneumonia|respirat|pulmonary|\blung\b|emphysema|bronch|\bcopd\b|asthma|\bards\b|airway|pleural"
                     r"|thoracotomy|thoracic|trache|pneumothorax|hemothorax|embolus|aspiration|apnea|ventilat|hypoxi"
                     r"|hypercarb|lobectomy|pneumonectomy|laryng|epiglott|inhalation|drowning"),
]
# PHENOTYPE_OVERRIDES = {}   # {"libellé exact de apacheadmissiondx": "phénotype"} : corrections manuelles, à figer avant E1

PHENOTYPE_OVERRIDES = {   # corrections relues le 2026-10-26 (libellés exacts de phenotype_map.csv)
    # -> autre
    'Encephalopathy, hepatic': 'autre',
    'Diabetic hyperglycemic hyperosmolar nonketotic coma (HHNC)': 'autre',
    'Heat exhaustion/stroke': 'autre',
    'Bleeding, GI from esophageal varices/portal hypertension': 'autre',
    'GI vascular ischemia, surgery for (resection)': 'autre',
    'GI vascular insufficiency': 'autre',
    # -> neurologique
    'Transphenoidal surgery': 'neurologique',
    'Cranioplasty and complications from previous craniotomies': 'neurologique',
    'Burr hole placement': 'neurologique',
    'Arteriovenous malformation, surgery for': 'neurologique',
    'Shunts and revisions': 'neurologique',
    'Stereotactic procedure': 'neurologique',
    'Amyotrophic lateral sclerosis': 'neurologique',
    'Devices for spine fracture/dislocation': 'neurologique',
    # -> cardiovasculaire
    'Thrombectomy (with general anesthesia)': 'cardiovasculaire',
    'Thrombectomy (without general anesthesia)': 'cardiovasculaire',
    'Embolectomy (with general anesthesia)': 'cardiovasculaire',
    'Embolectomy (without general anesthesia)': 'cardiovasculaire',
    'Monitoring, hemodynamic (pre-operative evaluation)': 'cardiovasculaire',
    'Complications of previous open-heart surgery, surgery for (i.e. bleeding, infection, mediastinal rewiring,leaking aortic graft etc.)': 'cardiovasculaire',
    'Complications of previous open heart surgery (i.e. bleeding, infection etc.)': 'cardiovasculaire',
    'Complications of prev. peripheral vasc. surgery,surgery for (i.e.ligation of bleeder, exploration and evacuation of hematoma, debridement, pseudoaneurysms, clots, fistula, etc.)': 'cardiovasculaire',
    'Heart transplant': 'cardiovasculaire',
    'Atrial Septal Defect (ASD) Repair': 'cardiovasculaire',
    'Congenital Defect Repair (Other)': 'cardiovasculaire',
    'Papillary muscle rupture': 'cardiovasculaire',
    'Vena cava filter insertion': 'cardiovasculaire',
    'Vena cava clipping': 'cardiovasculaire',
    # -> respiratoire
    'Atelectasis': 'respiratoire',
    'Bullectomy': 'respiratoire',
}

OPERATIVE_SOURCES = {"Operating Room", "Recovery Room", "PACU"}
SURGICAL_DX = r"surgery|\bcabg\b|replacement|endarterectomy|laminectomy|thoracotomy|craniotomy|repair|resection|ectomy|graft|bypass|transplant"

def dx_label(s):
    return s.fillna("").astype(str).str.strip().replace("", "missing")

def phenotype_of(dx):
    """Renvoie (phénotype, motif trouvé)."""
    if dx == "missing":
        return "autre", "manquant"
    clean = re.sub(r"\(.*?\)", " ", dx).lower()
    for name, pattern in PHENOTYPE_RULES:
        m = re.search(pattern, clean)
        if m:
            return name, m.group(0)
    return "autre", ""

assert set(PHENOTYPE_OVERRIDES.values()) <= set(PHENOTYPES)
stays["dx"] = dx_label(stays["apacheadmissiondx"])
dx_map = (stays.groupby("dx").agg(n_stays=("patientunitstayid", "size"), n_deaths=("y_mortality", "sum"))
          .reset_index())
res = [phenotype_of(d) for d in dx_map["dx"]]
dx_map["phenotype"] = [r[0] for r in res]
dx_map["rule"] = [r[1] for r in res]
ov = dx_map["dx"].isin(list(PHENOTYPE_OVERRIDES))
dx_map.loc[ov, "phenotype"] = dx_map.loc[ov, "dx"].map(PHENOTYPE_OVERRIDES)
dx_map.loc[ov, "rule"] = "override"
dx_map = dx_map.sort_values(["n_stays", "dx"], ascending=[False, True]).reset_index(drop=True)
assert dx_map["phenotype"].isin(PHENOTYPES).all()

stays["phenotype"] = stays["dx"].map(dx_map.set_index("dx")["phenotype"])
stays["operative"] = stays["unitadmitsource"].isin(OPERATIVE_SOURCES).astype("int8")
stays["surgical_dx"] = stays["dx"].str.contains(SURGICAL_DX, case=False, regex=True).astype("int8")
assert stays["phenotype"].notna().all()

# ---- résumé ----
ph = stays.groupby("phenotype").agg(sejours=("patientunitstayid", "size"), deces=("y_mortality", "sum"),
                                    mortalite=("y_mortality", "mean"), operatoire=("operative", "mean"))
ph["part"] = ph["sejours"] / len(stays)
ph["hopitaux_avec_10_deces"] = (stays[stays["y_mortality"] == 1].groupby(["phenotype", "hospitalid"]).size()
                                .ge(10).groupby(level=0).sum())
display(ph.reindex(PHENOTYPES).round(3))
print("Décès par phénotype et par strate :")
display(pd.crosstab(stays["phenotype"], stays["stratum"], values=stays["y_mortality"], aggfunc="sum")
        .reindex(index=PHENOTYPES, columns=list(STRATA)))
print(f"{len(dx_map)} diagnostics distincts. Correspondance des 60 plus fréquents (à relire) :")
display(dx_map.head(60)[["dx", "n_stays", "n_deaths", "phenotype", "rule"]])
print("Diagnostics les plus fréquents classés « autre » (vérifier qu'aucun ne devrait être ailleurs) :")
display(dx_map[dx_map["phenotype"] == "autre"].head(30)[["dx", "n_stays", "n_deaths", "rule"]])
print("Concordance entre l'axe opératoire (provenance) et un motif chirurgical du diagnostic :")
display(pd.crosstab(stays["operative"], stays["surgical_dx"], margins=True))

,sejours,deces,mortalite,operatoire,part,hopitaux_avec_10_deces
phenotype,,,,,,
sepsis,10323,1789,0.173,0.004,0.129,65
cardiovasculaire,25633,2177,0.085,0.348,0.321,76
neurologique,14501,1250,0.086,0.186,0.182,40
respiratoire,10252,1310,0.128,0.109,0.128,46
autre,19164,1089,0.057,0.256,0.240,43


Décès par phénotype et par strate :


stratum,petit,moyen,grand
phenotype,,,
sepsis,421,876,492
cardiovasculaire,531,924,722
neurologique,184,541,525
respiratoire,313,589,408
autre,256,469,364


389 diagnostics distincts. Correspondance des 60 plus fréquents (à relire) :


,dx,n_stays,n_deaths,phenotype,rule
0,"Sepsis, pulmonary",3878,792,sepsis,sepsis
1,"Infarction, acute myocardial (MI)",3750,193,cardiovasculaire,infarction
2,"CVA, cerebrovascular accident/stroke",3733,379,neurologique,cva
3,"CABG alone, coronary artery bypass grafting",2960,35,cardiovasculaire,cabg
4,"CHF, congestive heart failure",2745,326,cardiovasculaire,chf
5,"Sepsis, renal/UTI (including bladder)",2407,290,sepsis,sepsis
6,Cardiac arrest (with or without respiratory arrest; for respiratory arrest s...,2241,1004,cardiovasculaire,cardiac
7,"Pneumonia, bacterial",1751,277,respiratoire,pneumonia
8,Emphysema/bronchitis,1593,138,respiratoire,emphysema
9,"Rhythm disturbance (atrial, supraventricular)",1548,109,cardiovasculaire,rhythm


Diagnostics les plus fréquents classés « autre » (vérifier qu'aucun ne devrait être ailleurs) :


,dx,n_stays,n_deaths,rule
11,"Bleeding, upper GI",1471,87,
12,Diabetic ketoacidosis,1462,15,
19,"Bleeding, lower GI",1055,42,
20,"Renal failure, acute",1015,77,
22,"Bleeding, GI-location unknown",902,55,
30,Acid-base/electrolyte disturbance,632,41,
35,"Overdose, sedatives, hypnotics, antipsychotics, benzodiazepines",504,5,
36,"GI obstruction, surgery for (including lysis of adhesions)",503,54,
37,"Hypovolemia (including dehydration, Do not include shock states)",486,50,
38,"GI perforation/rupture, surgery for",478,58,


Concordance entre l'axe opératoire (provenance) et un motif chirurgical du diagnostic :


surgical_dx,0,1,All
operative,,,
0,61434,755,62189
1,2984,14700,17684
All,64418,15455,79873


## Étape 3 — Variables tabulaires des 24 heures et disponibilité

Les tenseurs bruts d'OP-FL (`X_raw`, `M`, `C`, `S_num`) sont résumés par séjour. Les variables asymétriques (même liste qu'OP-FL) passent d'abord par `log1p`.

| Bloc | Variables produites | Type |
|---|---|---|
| Pour chacune des 29 variables temporelles | moyenne, minimum, maximum, dernière valeur (`NaN` si jamais mesurée) | continue |
| | nombre d'observations sur 24 h (`log1p` de la somme de `C`) | continue |
| | observée au moins une fois (`M`, inclut le GCS « non évaluable ») | binaire |
| Diurèse | volume total sur 24 h (`log1p`) | continue |
| Statiques (`S_num` d'OP-FL) | âge, taille, poids ; sexe ; indicateurs de valeur manquante | continue / binaire |
| Catégorielles | 20 diagnostics d'admission les plus fréquents + autre + manquant ; phénotype ; type d'unité ; provenance (urgences, bloc, service, admission directe, autre hôpital, autre réanimation, autre) | one-hot |

* **Vocabulaires des catégories** (20 diagnostics, types d'unité) : choisis sur les **hôpitaux de réglage** seulement, à partir de fréquences, sans label.
* **Rien n'est normalisé ici.** La normalisation globale est recalculée au chargement (`src/tdtfl_data.py`), pour chaque régime et chaque graine, sur le seul split d'entraînement des hôpitaux de la fédération. Elle est **obligatoire** pour le transfert : appliqué aux patients de j, le modèle de i doit recevoir des entrées à la même échelle que celles sur lesquelles il a appris.
* **Disponibilité** (`avail`) : une variable est disponible dans un hôpital si au moins 5 % de ses séjours l'observent. C'est une propriété du système d'information de l'hôpital (module utilisé ou non) : elle est calculée sur tous ses séjours, sans label.
* **Harmonisation** : une variable indisponible chez un hôpital est remise à « manquant » dans ses propres données. Et quand on applique le modèle de i chez j, les variables indisponibles chez i sont aussi remises à « manquant » chez j : le modèle de i ne reçoit jamais une entrée qu'il n'a pas pu apprendre.

In [15]:
# ============================================================
# ÉTAPE 3 — VARIABLES TABULAIRES (RÉSUMÉS 24 H) + DISPONIBILITÉ PAR HÔPITAL
# ============================================================
feats = load_json(OPFL_DIR / "features.json")
VARS = feats["features"]
T = int(feats["T"])
F = len(VARS)
log_mask = np.asarray(feats["log"], dtype=bool)
var_group = dict(zip(VARS, feats["group"]))
S_NUM = feats["static_num"]
assert S_NUM[:4] == ["age", "height", "weight", "gender_female"], S_NUM

def slug(v):
    return re.sub(r"[^0-9a-z]+", "_", str(v).lower()).strip("_")

SLUG = {v: slug(v) for v in VARS}
assert len(set(SLUG.values())) == F
TEMPORAL_STATS = ["mean", "min", "max", "last", "n_obs", "observed"]
URINE = "urine_output" if "urine_output" in VARS else None

ADMIT_GROUPS = {"Emergency Department": "urgences", "Chest Pain Center": "urgences",
                "Operating Room": "bloc", "Recovery Room": "bloc", "PACU": "bloc",
                "Floor": "service", "Acute Care/Floor": "service", "Step-Down Unit (SDU)": "service",
                "ICU to SDU": "service", "Observation": "service", "Direct Admit": "admission_directe",
                "Other Hospital": "autre_hopital", "Other ICU": "autre_rea", "ICU": "autre_rea"}
ADMIT_LEVELS = ["urgences", "bloc", "service", "admission_directe", "autre_hopital", "autre_rea", "autre"]

# ---- vocabulaires choisis sur les hôpitaux de réglage (sans label) ----
tune_stays = stays[stays["role"] == "tune"]
DX_TOP = tune_stays.loc[tune_stays["dx"] != "missing", "dx"].value_counts().head(N_TOP_DX).index.tolist()
unit_share = tune_stays["unittype"].fillna("missing").value_counts(normalize=True)
UNIT_LEVELS = sorted(u for u, sh in unit_share.items() if sh >= MIN_UNIT_SHARE and u != "missing")

# ---- métadonnées des colonnes ----
meta = []
for v in VARS:
    for st in TEMPORAL_STATS:
        meta.append({"name": f"{SLUG[v]}__{st}", "var": v, "stat": st,
                     "kind": "bin" if st == "observed" else "cont", "group": var_group[v], "label": v})
if URINE:
    meta.append({"name": "urine_output__total", "var": URINE, "stat": "total", "kind": "cont",
                 "group": var_group[URINE], "label": "diurèse totale 24 h"})
for name, kind, label in [("age", "cont", "âge"), ("height", "cont", "taille"), ("weight", "cont", "poids"),
                          ("gender_female", "bin", "sexe féminin"), ("height_missing", "bin", "taille manquante"),
                          ("weight_missing", "bin", "poids manquant"), ("gender_missing", "bin", "sexe manquant")]:
    meta.append({"name": name, "var": "static", "stat": "static", "kind": kind, "group": "static", "label": label})
DX_COLS = [f"dx__{i:02d}" for i in range(len(DX_TOP))]
for c, d in zip(DX_COLS, DX_TOP):
    meta.append({"name": c, "var": "dx", "stat": "onehot", "kind": "bin", "group": "dx", "label": d})
meta += [{"name": "dx__autre", "var": "dx", "stat": "onehot", "kind": "bin", "group": "dx", "label": "autre diagnostic"},
         {"name": "dx__manquant", "var": "dx", "stat": "onehot", "kind": "bin", "group": "dx", "label": "diagnostic manquant"}]
for p in PHENOTYPES:
    meta.append({"name": f"pheno__{p}", "var": "phenotype", "stat": "onehot", "kind": "bin", "group": "phenotype", "label": p})
for u in UNIT_LEVELS + ["autre"]:
    meta.append({"name": f"unit__{slug(u)}", "var": "unittype", "stat": "onehot", "kind": "bin", "group": "unit", "label": u})
for a in ADMIT_LEVELS:
    meta.append({"name": f"admit__{a}", "var": "admitsource", "stat": "onehot", "kind": "bin", "group": "admit", "label": a})
meta = pd.DataFrame(meta)
assert meta["name"].is_unique

def summarize(d):
    """Résumés 24 h d'un hôpital (n séjours) à partir des tenseurs bruts d'OP-FL."""
    raw = d["X_raw"].astype(np.float64)
    x = raw.copy()
    x[..., log_mask] = np.log1p(np.clip(x[..., log_mask], 0, None))
    has = ~np.isnan(x)
    any_val = has.any(axis=1)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        mean, mn, mx = np.nanmean(x, axis=1), np.nanmin(x, axis=1), np.nanmax(x, axis=1)
    last_t = T - 1 - np.argmax(has[:, ::-1, :], axis=1)
    last = np.take_along_axis(x, last_t[:, None, :], axis=1)[:, 0, :]
    last[~any_val] = np.nan
    n_obs = np.log1p(d["C"].astype(np.float64).sum(axis=1))
    observed = d["M"].any(axis=1)
    out = {}
    for f, v in enumerate(VARS):
        s = SLUG[v]
        out[f"{s}__mean"], out[f"{s}__min"], out[f"{s}__max"] = mean[:, f], mn[:, f], mx[:, f]
        out[f"{s}__last"], out[f"{s}__n_obs"] = last[:, f], n_obs[:, f]
        out[f"{s}__observed"] = observed[:, f].astype(np.float64)
    if URINE:
        f = VARS.index(URINE)
        tot = np.nansum(raw[:, :, f], axis=1)
        tot[~any_val[:, f]] = np.nan
        out["urine_output__total"] = np.log1p(np.clip(tot, 0, None))
    S = d["S_num"].astype(np.float64)
    out["age"], out["height"], out["weight"] = S[:, 0], S[:, 1], S[:, 2]
    out["gender_female"] = np.nan_to_num(S[:, 3], nan=0.0)
    out["height_missing"] = np.isnan(S[:, 1]).astype(np.float64)
    out["weight_missing"] = np.isnan(S[:, 2]).astype(np.float64)
    out["gender_missing"] = np.isnan(S[:, 3]).astype(np.float64)
    return pd.DataFrame(out), observed.mean(axis=0)

# ---- passage sur les hôpitaux retenus ----
st_idx = stays.set_index("patientunitstayid")
frames, coverage = [], {}
for i, h in enumerate(ELIGIBLE):
    d = np.load(OPFL_DIR / "clients" / f"hospital_{h}.npz")
    sid = d["stay_id"]
    expected = stays.loc[stays["hospitalid"] == h, "patientunitstayid"]
    assert set(sid) == set(expected), f"hôpital {h} : séjours du .npz différents de cohort_final"
    assert d["X_raw"].shape[1:] == (T, F), f"hôpital {h} : forme inattendue {d['X_raw'].shape}"
    assert np.array_equal(d["y_mortality"].astype(int), st_idx.loc[sid, "y_mortality"].to_numpy().astype(int)), \
        f"hôpital {h} : mortalité du .npz différente de cohort_final"
    df, cov = summarize(d)
    df.insert(0, "patientunitstayid", sid)
    frames.append(df)
    coverage[int(h)] = cov
    print(f"  résumés : hôpital {i + 1}/{len(ELIGIBLE)}", end="\r")
print()
X = pd.concat(frames, ignore_index=True).set_index("patientunitstayid").loc[stays["patientunitstayid"]]

# ---- catégorielles (depuis la cohorte) ----
dx_pos = {d: i for i, d in enumerate(DX_TOP)}
dx_idx = stays["dx"].map(dx_pos)
cat = {}
for i, c in enumerate(DX_COLS):
    cat[c] = (dx_idx == i).to_numpy()
cat["dx__manquant"] = (stays["dx"] == "missing").to_numpy()
cat["dx__autre"] = (dx_idx.isna() & (stays["dx"] != "missing")).to_numpy()
for p in PHENOTYPES:
    cat[f"pheno__{p}"] = (stays["phenotype"] == p).to_numpy()
unit = stays["unittype"].where(stays["unittype"].isin(UNIT_LEVELS), "autre")
for u in UNIT_LEVELS + ["autre"]:
    cat[f"unit__{slug(u)}"] = (unit == u).to_numpy()
admit = stays["unitadmitsource"].map(ADMIT_GROUPS).fillna("autre")
for a in ADMIT_LEVELS:
    cat[f"admit__{a}"] = (admit == a).to_numpy()
cat = pd.DataFrame({k: v.astype(np.float64) for k, v in cat.items()}, index=X.index)
features = pd.concat([X, cat], axis=1)[meta["name"].tolist()].astype(np.float32)
assert len(features) == len(stays) and features.index.is_unique
assert not np.isinf(features.to_numpy()).any()

# ---- disponibilité ----
cov_df = pd.DataFrame(coverage, index=VARS).T.sort_index()
cov_df.index.name = "hospitalid"
avail = (cov_df >= AVAIL_MIN_COVERAGE).astype(int)

# (les fichiers sont écrits à l'étape 7, seulement si le protocole est validé)

# ---- résumé ----
print(f"{len(features)} séjours × {features.shape[1]} variables :", meta.groupby("kind").size().to_dict())
print("Variables par groupe :", meta.groupby("group").size().to_dict())
na = pd.DataFrame({"manquant_moyenne_24h": [features[f"{SLUG[v]}__mean"].isna().mean() for v in VARS],
                   "hopitaux_indisponibles": [int((avail[v] == 0).sum()) for v in VARS]}, index=VARS)
display(na.round(3))

A = avail.loc[ELIGIBLE, VARS].to_numpy().astype(bool)
masked = (~A).astype(int) @ A.T.astype(int)          # masked[i, j] = variables dispo chez j mais pas chez i
off = ~np.eye(len(A), dtype=bool)
print(f"Paires dirigées (source i, cible j) : {off.sum()} ; avec au moins une variable remise à manquant : "
      f"{(masked[off] > 0).mean():.1%} ; nombre moyen de variables masquées : {masked[off].mean():.2f}")
print("Diagnostics codés en one-hot :", DX_TOP[:5], "...")
print("Types d'unité codés :", UNIT_LEVELS)

  résumés : hôpital 101/101
79873 séjours × 225 variables : {'bin': 76, 'cont': 149}
Variables par groupe : {'admit': 7, 'bloodgas': 30, 'dx': 22, 'gcs': 30, 'monitor': 36, 'phenotype': 5, 'routine': 66, 'static': 7, 'temperature': 6, 'unit': 9, 'urine': 7}


,manquant_moyenne_24h,hopitaux_indisponibles
lactate,0.725,9
creatinine,0.079,0
BUN,0.083,0
sodium,0.079,0
potassium,0.069,0
bicarbonate,0.137,5
glucose,0.086,0
WBC x 1000,0.108,0
Hgb,0.096,0
Hct,0.093,0


Paires dirigées (source i, cible j) : 10100 ; avec au moins une variable remise à manquant : 40.7% ; nombre moyen de variables masquées : 1.39
Diagnostics codés en one-hot : ['Sepsis, pulmonary', 'Infarction, acute myocardial (MI)', 'CABG alone, coronary artery bypass grafting', 'CVA, cerebrovascular accident/stroke', 'CHF, congestive heart failure'] ...
Types d'unité codés : ['CCU-CTICU', 'CSICU', 'CTICU', 'Cardiac ICU', 'MICU', 'Med-Surg ICU', 'Neuro ICU', 'SICU']


## Étape 4 — Découpages R1 (régime statique), 10 graines

Dans chaque hôpital, les séjours sont répartis **train 60 % / D 20 % / test 20 %**, stratifiés sur **décès × phénotype**, avec une nouvelle répartition à chaque graine. Les rôles des hôpitaux (réglage, évaluation) restent fixes.

Méthode (`blocked_allocation`) : on range les séjours par strate (survivants puis décès, phénotype par phénotype, ordre aléatoire à l'intérieur), puis on découpe cette liste en blocs de 5 qui reçoivent chacun une permutation aléatoire de (train, train, train, D, test). Chaque strate reçoit ainsi ses proportions à un ou deux patients près. C'est important ici : un petit hôpital n'a qu'environ 7 décès dans D, et un tirage indépendant en aurait souvent 4 ou 10.

Le train reçoit en plus **5 plis internes** (même méthode), pour régler la régularisation et produire des prédictions hors échantillon (calibration) sans toucher à D.

| Partie | Usage |
|---|---|
| train | entraîner le modèle local ; recalibrer (Platt) le modèle d'un partenaire sur les patients de la cible |
| D | décider : choisir le partenaire, autoriser ou refuser le transfert |
| test | mesurer, et rien d'autre |

In [16]:
# ============================================================
# ÉTAPE 4 — DÉCOUPAGES R1 : TRAIN / D / TEST + PLIS INTERNES, 10 GRAINES
# ============================================================
PH_CODE = {p: i for i, p in enumerate(PHENOTYPES)}
y_all = stays["y_mortality"].to_numpy().astype(int)
strat_key = y_all * 10 + stays["phenotype"].map(PH_CODE).to_numpy().astype(int)
hosp_arr = stays["hospitalid"].to_numpy()
HOSP_ROWS = {int(h): np.flatnonzero(hosp_arr == h) for h in ELIGIBLE}

splits = pd.DataFrame({"patientunitstayid": stays["patientunitstayid"].to_numpy()})
for s in range(N_SEEDS):
    r1 = np.full(len(stays), -1, dtype=np.int8)
    inner = np.full(len(stays), -1, dtype=np.int8)
    for h, rows in HOSP_ROWS.items():
        rng = rng_for(1, s, h)
        lab = blocked_allocation(strat_key[rows], R1_PATTERN, rng)
        r1[rows] = lab
        tr = rows[lab == R1_CODES["train"]]
        inner[tr] = blocked_allocation(strat_key[tr], list(range(N_INNER_FOLDS)), rng)
    splits[f"r1_s{s}"] = r1
    splits[f"r1_inner_s{s}"] = inner

# ---- résumé (graine 0) ----
tmp = stays[["hospitalid", "stratum", "y_mortality"]].assign(part=splits["r1_s0"].map({v: k for k, v in R1_CODES.items()}))
per_h = tmp.groupby(["stratum", "hospitalid", "part"])["y_mortality"].agg(["size", "sum"]).unstack("part")
print("Graine 0 — séjours et décès moyens par hôpital, par strate :")
display(per_h.groupby(level="stratum").mean().round(1).reindex(list(STRATA)))
print("Proportions globales (graine 0) :", tmp["part"].value_counts(normalize=True).round(4).to_dict())

Graine 0 — séjours et décès moyens par hôpital, par strate :


size                  sum             
part         D   test   train     D  test  train
stratum                                         
petit     82.3   82.4   247.3   6.8   6.7   20.6
moyen    174.5  174.5   523.5  17.0  16.8   51.1
grand    443.2  443.4  1329.6  45.6  45.8  136.8

Proportions globales (graine 0) : {'train': 0.6001, 'test': 0.2, 'D': 0.1999}


## Étape 5 — Découpages R2 (régime temporel)

Seuls les hôpitaux `temporal` (≥ 10 décès chaque année) participent, comme sources **et** comme cibles : aucun modèle ne voit 2015 à l'entraînement.

| Année | Parties | Proportions |
|---|---|---|
| 2014 | train14 / D14 / test14 | 60 / 20 / 20 |
| 2015 | D15 / test15 | 50 / 50 |

* **R2** : modèles entraînés sur train14, décisions prises sur D14, évaluation sur test15. L'hypothèse A2 (même période pour D et le test) est volontairement violée : on mesure ce que devient la sécurité quand elle n'est plus garantie. test14 donne la référence sans changement de période.
* **R2b** : flux de lots D14 puis D15 ; chaque lot est évalué sur le test de la même période (test14 puis test15). Le passage à 2015 est un vrai point de changement.

In [17]:
# ============================================================
# ÉTAPE 5 — DÉCOUPAGES R2 : TEMPOREL 2014 -> 2015
# ============================================================
year_arr = stays["year"].to_numpy()
temporal_ids = [int(h) for h in ELIGIBLE if hospitals.loc[h, "temporal"]]
for s in range(N_SEEDS):
    r2 = np.full(len(stays), -1, dtype=np.int8)
    for h in temporal_ids:
        rows = HOSP_ROWS[h]
        rng = rng_for(2, s, h)
        for yr, pattern in ((YEARS[0], R2_PATTERN_Y1), (YEARS[1], R2_PATTERN_Y2)):
            sub = rows[year_arr[rows] == yr]
            r2[sub] = blocked_allocation(strat_key[sub], pattern, rng)
    splits[f"r2_s{s}"] = r2

inv2 = {v: k for k, v in R2_CODES.items()}
tmp = stays[["hospitalid", "stratum", "y_mortality"]].assign(part=splits["r2_s0"].map(inv2))
tmp = tmp[tmp["part"].notna()]
print(f"Régime temporel : {len(temporal_ids)} hôpitaux, {len(tmp)} séjours")
print("Graine 0 — décès moyens par hôpital et par partie :")
display(tmp.groupby(["stratum", "hospitalid", "part"])["y_mortality"].sum().unstack("part")
        .groupby(level="stratum").mean().round(1).reindex(list(STRATA))[list(R2_CODES)])

Régime temporel : 88 hôpitaux, 74468 séjours
Graine 0 — décès moyens par hôpital et par partie :


part,train14,D14,test14,D15,test15
stratum,,,,,
petit,11.3,3.6,3.8,8.4,8.8
moyen,26.1,8.8,8.9,21.2,20.9
grand,70.1,23.6,23.5,55.4,55.7


## Étape 6 — Régime R3 (hôpital non vu)

L'hôpital non vu n'entraîne pas de modèle. Ce qui lui est disponible est défini explicitement :

| Variante | Labels chez la cible | Ce que fait la méthode |
|---|---|---|
| R3a | aucun | rien ne peut être décidé : on applique le modèle global de la fédération (référence) |
| R3b | budget b ∈ {50, 100, 200} patients étiquetés | b/2 servent à recalibrer (Platt) le modèle global et ceux des partenaires, b/2 forment D |

* Pour chaque graine, le **test** de l'hôpital non vu est son test R1. Ses autres séjours (train et D de R1) forment un réservoir d'étiquettes, coupé en deux **réservoirs** (calibration, D) stratifiés sur décès × phénotype.
* Dans chaque réservoir, un **ordre d'acquisition** garde la proportion de décès dans tout préfixe : le budget b prend les b/2 premiers patients de chaque réservoir. Les budgets sont donc emboîtés.
* **Fédération** : pour une cible du pli k, les sources et le modèle global sont entraînés sur les hôpitaux hors du pli k (split train R1). La normalisation globale est recalculée de même.

In [18]:
# ============================================================
# ÉTAPE 6 — R3 : RÉSERVOIRS D'ÉTIQUETAGE DE L'HÔPITAL NON VU
# ============================================================
for s in range(N_SEEDS):
    pool = np.full(len(stays), -1, dtype=np.int8)
    rank = np.full(len(stays), -1, dtype=np.int32)
    r1 = splits[f"r1_s{s}"].to_numpy()
    for h, rows in HOSP_ROWS.items():
        rng = rng_for(3, s, h)
        nontest = rows[r1[rows] != R1_CODES["test"]]
        p = blocked_allocation(strat_key[nontest], [R3_POOLS["cal"], R3_POOLS["D"]], rng)
        pool[nontest] = p
        for code in R3_POOLS.values():
            sel = nontest[p == code]
            rank[sel] = interleaved_rank(y_all[sel], rng)
    splits[f"r3_pool_s{s}"] = pool
    splits[f"r3_rank_s{s}"] = rank

# ---- résumé (graine 0) : budgets réalisables et décès attendus dans D ----
pool0 = splits["r3_pool_s0"].to_numpy()
rows_tab = []
for h, rows in HOSP_ROWS.items():
    n_pool = min(int((pool0[rows] == 0).sum()), int((pool0[rows] == 1).sum()))
    rec = {"hospitalid": h, "stratum": hospitals.loc[h, "stratum"], "reservoir_min": n_pool}
    for b in R3_BUDGETS:
        rec[f"b{b}_possible"] = n_pool >= b - b // 2
        rec[f"b{b}_deces_D"] = hospitals.loc[h, "mortality"] * (b - b // 2)
    rows_tab.append(rec)
r3_tab = pd.DataFrame(rows_tab)
agg = {"reservoir_min": "median"}
agg.update({f"b{b}_possible": "mean" for b in R3_BUDGETS})
agg.update({f"b{b}_deces_D": "mean" for b in R3_BUDGETS})
print("Part des hôpitaux qui peuvent fournir chaque budget, et décès attendus dans D :")
display(r3_tab.groupby("stratum").agg(agg).reindex(list(STRATA)).round(2))

Part des hôpitaux qui peuvent fournir chaque budget, et décès attendus dans D :


,reservoir_min,b50_possible,b100_possible,b200_possible,b50_deces_D,b100_deces_D,b200_deces_D
stratum,,,,,,,
petit,142.0,1.0,1.0,0.88,2.31,4.62,9.23
moyen,324.5,1.0,1.0,1.00,2.68,5.36,10.72
grand,782.0,1.0,1.0,1.00,2.65,5.30,10.59


## Étape 7 — Protocole figé

Le hash couvre la table des hôpitaux (strates, rôles, régime temporel, plis R3), la correspondance des phénotypes et tous les découpages. Si `protocol_tdtfl.json` existe déjà, la cellule **vérifie** que le tirage est identique au lieu de l'écraser ; en cas de différence, elle s'arrête **avant d'écrire quoi que ce soit**.

Les variables tabulaires ne font pas partie du hash (elles ne contiennent aucun tirage) : elles sont réécrites à chaque exécution validée, et leur empreinte est suivie dans `feature_info.json`.

In [19]:
# ============================================================
# ÉTAPE 7 — PROTOCOLE FIGÉ : HASH, FICHIERS DU PROTOCOLE
# ============================================================
HOSP_COLS = (["n_stays", "n_deaths", "mortality"] + [f"{a}_{y}" for a in ("n", "deaths") for y in YEARS]
             + ["stratum", "role", "temporal", "r3_fold", "opfl_role"])
hosp_table = hospitals.loc[ELIGIBLE, HOSP_COLS].sort_index()
hosp_table.index.name = "hospitalid"
splits = splits.sort_values("patientunitstayid").reset_index(drop=True)
map_table = dx_map[["dx", "n_stays", "n_deaths", "phenotype", "rule"]]

components = {"hospitals": digest_df(hosp_table.reset_index()),
              "phenotype_map": digest_df(map_table[["dx", "phenotype"]].sort_values("dx")),
              "splits": digest_df(splits)}
digest = hashlib.sha256("|".join(f"{k}={v}" for k, v in sorted(components.items())).encode()).hexdigest()[:16]
features_fingerprint = hashlib.sha256(meta.to_csv(index=False, lineterminator="\n").encode()
                                      + pd.util.hash_pandas_object(features, index=True).to_numpy().tobytes()).hexdigest()[:16]

STAY_COLS = ["patientunitstayid", "uniquepid", "hospitalid", "year", "dx", "phenotype", "operative", "surgical_dx",
             "unittype", "unitadmitsource", "age_num", "gender", "ethnicity", "y_mortality"] + out_cols + \
            ["stratum", "role", "temporal", "r3_fold"]
stays_out = stays[STAY_COLS].sort_values("patientunitstayid").reset_index(drop=True)

protocol = {
    "name": "TDT-FL", "version": PROTOCOL_VERSION, "hash": digest, "components": components,
    "source": {"opfl_data_dir": str(OPFL_DIR), "opfl_protocol_sha256": OPFL_PROTOCOL_SHA,
               "inputs": ["interim/cohort_final.parquet", "interim/outcomes.parquet", "features.json",
                          "clients/hospital_<id>.npz"]},
    "seeds": {"base_seed": BASE_SEED, "n_seeds": N_SEEDS},
    "hospitals": {"min_deaths": MIN_DEATHS, "strata": STRATA, "tune_fraction": TUNE_FRACTION,
                  "n_eligible": int(len(ELIGIBLE)),
                  "tune": [int(h) for h in hosp_table.index[hosp_table["role"] == "tune"]],
                  "eval": [int(h) for h in hosp_table.index[hosp_table["role"] == "eval"]],
                  "temporal": temporal_ids, "min_deaths_per_year": MIN_DEATHS_PER_YEAR,
                  "r3_folds": {str(k): [int(h) for h in hosp_table.index[hosp_table["r3_fold"] == k]]
                               for k in range(N_R3_FOLDS)}},
    "phenotypes": {"order": PHENOTYPES, "rules": PHENOTYPE_RULES, "overrides": PHENOTYPE_OVERRIDES,
                   "parentheses_removed": True, "operative_sources": sorted(OPERATIVE_SOURCES)},
    "regimes": {
        "r1": {"codes": R1_CODES, "ratios": {"train": 0.6, "D": 0.2, "test": 0.2},
               "stratification": "décès x phénotype, allocation par blocs", "inner_folds": N_INNER_FOLDS},
        "r2": {"codes": R2_CODES, "years": list(YEARS), "ratios": {"2014": "60/20/20", "2015": "50/50"},
               "hospitals": "temporal (sources et cibles)"},
        "r3": {"pools": R3_POOLS, "budgets": list(R3_BUDGETS), "test": "test r1 de la même graine",
               "federation": "hôpitaux hors du pli r3 de la cible"},
        "r4": "dérive simulée : lots tirés des réservoirs D et test de R1, journal des changements (spécification)"},
    "usage": {"train": "entraînement du modèle local ; recalibration de Platt des partenaires sur la cible",
              "D": "décisions uniquement (choix du partenaire, porte)", "test": "mesure uniquement"},
    "features": {"avail_min_coverage": AVAIL_MIN_COVERAGE, "n_top_dx": N_TOP_DX,
                 "vocabularies_from": "hôpitaux de réglage, fréquences sans label",
                 "normalization": "globale, au chargement, split train de la fédération du régime",
                 "harmonization": "variable indisponible chez la source -> manquante chez la cible",
                 "details": "feature_info.json"},
    "created_at": datetime.now().isoformat(timespec="seconds"),
}

proto_path = TDT_DIR / "protocol_tdtfl.json"
files = {"splits.parquet": lambda p: splits.to_parquet(p, index=False),
         "stays.parquet": lambda p: stays_out.to_parquet(p, index=False),
         "hospitals.csv": lambda p: hosp_table.to_csv(p),
         "phenotype_map.csv": lambda p: map_table.to_csv(p, index=False)}
if proto_path.exists():
    old = load_json(proto_path)
    if old["hash"] != digest:
        changed = [k for k in components if old.get("components", {}).get(k) != components[k]]
        raise RuntimeError(
            f"Le protocole existant (hash {old['hash']}) diffère du tirage actuel ({digest}) ; "
            f"parties modifiées : {changed}. Un paramètre, la cohorte ou la correspondance des phénotypes a changé. "
            "Ne supprimez protocol_tdtfl.json que si vous voulez volontairement re-tirer le protocole.")
    for name, write in files.items():
        if not (TDT_DIR / name).exists():
            write(TDT_DIR / name)
    print(f"Protocole existant retrouvé à l'identique (hash {digest}) : rien n'est écrasé.")
else:
    for name, write in files.items():
        write(TDT_DIR / name)
    save_json(protocol, proto_path)
    print(f"Protocole tiré et figé (hash {digest}).")
print("Composantes :", components)

# ---- variables tabulaires (après validation du protocole) ----
info_path = TDT_DIR / "feature_info.json"
if info_path.exists() and load_json(info_path).get("fingerprint") != features_fingerprint:
    print("Attention : les variables tabulaires diffèrent de la version précédente ; "
          "les résultats déjà calculés avec l'ancienne version ne sont plus comparables.")
features.reset_index().to_parquet(TDT_DIR / "features_raw.parquet", index=False)
meta.to_csv(TDT_DIR / "feature_meta.csv", index=False)
avail.to_csv(TDT_DIR / "availability.csv")
cov_df.round(4).to_csv(TDT_DIR / "coverage.csv")
save_json({"fingerprint": features_fingerprint, "protocol_hash": digest, "n_features": int(len(meta)),
           "kinds": meta.groupby("kind").size().to_dict(), "dx_top": DX_TOP, "unit_levels": UNIT_LEVELS,
           "admit_groups": ADMIT_GROUPS, "temporal_stats": TEMPORAL_STATS,
           "written_at": datetime.now().isoformat(timespec="seconds")}, info_path)
print(f"Variables écrites : {features.shape[1]} colonnes (empreinte {features_fingerprint}).")

Protocole tiré et figé (hash 0f91266774f979f1).
Composantes : {'hospitals': '554427c5e868f0e5', 'phenotype_map': 'ae97bc7dcd5429be', 'splits': 'ed234104361430ed'}
Attention : les variables tabulaires diffèrent de la version précédente ; les résultats déjà calculés avec l'ancienne version ne sont plus comparables.
Variables écrites : 225 colonnes (empreinte fe4582783af91509).


## Étape 8 — Module de chargement `src/tdtfl_data.py`

Cette cellule écrit le module que les notebooks suivants importeront ; il est la seule source de vérité pour l'encodage des données en entrée de modèle.

```python
data = tdtfl_data.TDTData(TDT_DIR)
norm = data.normalizer("r1", seed=0)                       # normalisation globale (train de la fédération)
dj   = data.matrix(j, "r1", 0, norm)                       # données de la cible j
dji  = data.matrix(j, "r1", 0, norm, also_mask=i)          # mêmes patients, prêts pour le modèle de i
train = tdtfl_data.select(dj, [tdtfl_data.R1["train"]])
```

In [20]:
# ============================================================
# ÉTAPE 8 — MODULE DE CHARGEMENT src/tdtfl_data.py
# ============================================================
TDTFL_MODULE = r'''
"""
tdtfl_data — chargement des données TDT-FL (protocole 1.0).

Généré par notebooks/02_protocole_tdtfl.ipynb (étape 8) : modifier le notebook, pas ce fichier.

Régimes et codes des découpages (graine s) :
- "r1" statique   : 0 train, 1 D, 2 test ; plis internes du train dans inner_folds(s) (0..4, -1 hors train).
- "r2" temporel   : hôpitaux `temporal` seulement ; 0 train14, 1 D14, 2 test14, 3 D15, 4 test15 (-1 sinon).
- "r3" non vu     : test = test r1 ; étiquettes de la cible = budget b pris dans deux réservoirs
                    (calibration, D) dans l'ordre d'acquisition (r3_labeled) ; fédération = hôpitaux hors
                    du pli r3 de la cible.

Encodage au chargement (rien n'est stocké normalisé) :
- continues : z-score avec la moyenne et l'écart-type du split train de la fédération
  (normalisation globale) ; valeur manquante -> 0, c'est-à-dire la moyenne ;
- binaires (indicateurs d'observation, one-hot, manquants) : 0/1 ;
- harmonisation : une variable indisponible dans un hôpital (avail = 0) est remise à « manquant » dans
  ses propres données ; matrix(j, ..., also_mask=i) la remet aussi à manquant chez j avant d'appliquer
  le modèle de i.
"""
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

R1 = {"train": 0, "D": 1, "test": 2}
R2 = {"train14": 0, "D14": 1, "test14": 2, "D15": 3, "test15": 4}
R3_POOLS = {"cal": 0, "D": 1}
MISSING_STATS = ("mean", "min", "max", "last", "total")   # -> NaN quand la variable est masquée
ZERO_STATS = ("n_obs", "observed")                        # -> 0 quand la variable est masquée


class TDTData:
    """Accès unique aux données TDT-FL. data = TDTData("../data_tdtfl")."""

    def __init__(self, tdt_dir):
        self.dir = Path(tdt_dir)
        self.protocol = json.loads((self.dir / "protocol_tdtfl.json").read_text(encoding="utf-8"))
        self.seeds = list(range(self.protocol["seeds"]["n_seeds"]))
        self.stays = pd.read_parquet(self.dir / "stays.parquet").set_index("patientunitstayid")
        self.hospitals = pd.read_csv(self.dir / "hospitals.csv", index_col="hospitalid")
        self.hospitals["temporal"] = self.hospitals["temporal"].astype(bool)
        self.meta = pd.read_csv(self.dir / "feature_meta.csv")
        self.columns = self.meta["name"].tolist()
        self.cont = (self.meta["kind"] == "cont").to_numpy()
        self.splits = pd.read_parquet(self.dir / "splits.parquet").set_index("patientunitstayid").loc[self.stays.index]
        raw = pd.read_parquet(self.dir / "features_raw.parquet").set_index("patientunitstayid")
        self.avail = pd.read_csv(self.dir / "availability.csv", index_col="hospitalid").astype(bool)
        hosp = self.stays["hospitalid"].to_numpy()
        self._rows = {int(h): np.flatnonzero(hosp == h) for h in np.unique(hosp)}
        x = raw.loc[self.stays.index, self.columns].to_numpy(dtype=np.float32, copy=True)
        for h, rows in self._rows.items():          # chaque hôpital : ses variables indisponibles -> manquant
            x[rows] = self._mask(x[rows], self.unavailable(h))
        self.raw = x

    # ------------------------------------------------------------------ hôpitaux
    def targets(self, regime="r1", role="eval", stratum=None):
        """Cibles d'un régime. role : 'eval' (résultats rapportés), 'tune' (réglage) ou None (toutes)."""
        h = self.hospitals
        keep = np.ones(len(h), dtype=bool)
        if regime == "r2":
            keep &= h["temporal"].to_numpy()
        if role is not None:
            keep &= (h["role"] == role).to_numpy()
        if stratum is not None:
            keep &= (h["stratum"] == stratum).to_numpy()
        return [int(i) for i in h.index[keep]]

    def federation(self, regime="r1", fold=None):
        """Hôpitaux qui entraînent un modèle (sources, modèle global) dans un régime."""
        h = self.hospitals
        if regime == "r1":
            return [int(i) for i in h.index]
        if regime == "r2":
            return [int(i) for i in h.index[h["temporal"].to_numpy()]]
        if regime == "r3":
            if fold is None:
                raise ValueError("régime r3 : préciser le pli de la cible (r3_fold(hôpital))")
            return [int(i) for i in h.index[h["r3_fold"].to_numpy() != int(fold)]]
        raise ValueError(f"régime inconnu : {regime}")

    def r3_fold(self, hospital_id):
        return int(self.hospitals.loc[int(hospital_id), "r3_fold"])

    def unavailable(self, hospital_id):
        a = self.avail.loc[int(hospital_id)]
        return a.index[~a.to_numpy()].tolist()

    def masked_variables(self, source, target):
        """Variables disponibles chez la cible mais pas chez la source (remises à manquant chez la cible)."""
        a_s, a_t = self.avail.loc[int(source)].to_numpy(), self.avail.loc[int(target)].to_numpy()
        return self.avail.columns[(~a_s) & a_t].tolist()

    # ---------------------------------------------------------------- découpages
    def codes(self, regime, seed):
        """Codes de partie de tous les séjours (ordre de self.stays)."""
        col = {"r1": f"r1_s{seed}", "r3": f"r1_s{seed}", "r2": f"r2_s{seed}"}[regime]
        return self.splits[col].to_numpy()

    def inner_folds(self, seed):
        return self.splits[f"r1_inner_s{seed}"].to_numpy()

    def r3_labeled(self, hospital_id, seed, budget):
        """Masques (calibration, D) sur les séjours de l'hôpital, pour un budget de `budget` patients étiquetés."""
        rows = self._rows[int(hospital_id)]
        pool = self.splits[f"r3_pool_s{seed}"].to_numpy()[rows]
        rank = self.splits[f"r3_rank_s{seed}"].to_numpy()[rows]
        n_cal = budget // 2
        n_d = budget - n_cal
        return (pool == R3_POOLS["cal"]) & (rank < n_cal), (pool == R3_POOLS["D"]) & (rank < n_d)

    def train_rows(self, regime, seed, fold=None):
        """Lignes du split d'entraînement de la fédération (les seules qui servent à la normalisation)."""
        rows = np.concatenate([self._rows[h] for h in self.federation(regime, fold)])
        return np.sort(rows[self.codes(regime, seed)[rows] == 0])

    # ------------------------------------------------------------------ encodage
    def normalizer(self, regime, seed, fold=None):
        """Normalisation globale : moyenne et écart-type des variables continues sur train_rows."""
        rows = self.train_rows(regime, seed, fold)
        x = self.raw[rows][:, self.cont].astype(np.float64)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            mean, std = np.nanmean(x, axis=0), np.nanstd(x, axis=0)
        mean = np.where(np.isfinite(mean), mean, 0.0)
        std = np.where(np.isfinite(std) & (std > 1e-6), std, 1.0)
        return {"regime": regime, "seed": int(seed), "fold": fold, "n_rows": int(len(rows)),
                "mean": mean.astype(np.float32), "std": std.astype(np.float32)}

    def _mask(self, x, variables):
        if not variables:
            return x
        m = self.meta["var"].isin(variables)
        x = x.copy()
        x[:, np.flatnonzero(m & self.meta["stat"].isin(MISSING_STATS))] = np.nan
        x[:, np.flatnonzero(m & self.meta["stat"].isin(ZERO_STATS))] = 0.0
        return x

    def encode(self, x, norm):
        x = x.copy()
        x[:, self.cont] = (x[:, self.cont] - norm["mean"]) / norm["std"]
        return np.nan_to_num(x, nan=0.0).astype(np.float32)

    def matrix(self, hospital_id, regime, seed, norm, also_mask=None):
        """Tous les séjours d'un hôpital, encodés. also_mask = hôpital source dont le modèle sera appliqué."""
        rows = self._rows[int(hospital_id)]
        x = self.raw[rows]
        if also_mask is not None:
            x = self._mask(x, self.unavailable(also_mask))
        st = self.stays.iloc[rows]
        out = {"hospital_id": int(hospital_id), "stay_id": st.index.to_numpy(),
               "split": self.codes(regime, seed)[rows], "X": self.encode(x, norm),
               "y_mortality": st["y_mortality"].to_numpy(dtype=np.float32),
               "phenotype": st["phenotype"].to_numpy(), "operative": st["operative"].to_numpy(),
               "year": st["year"].to_numpy()}
        for c in ("y_aki", "aki_ascertained", "apache_pred_mortality"):
            if c in st.columns:
                out[c] = st[c].to_numpy()
        if regime in ("r1", "r3"):
            out["inner_fold"] = self.inner_folds(seed)[rows]
        return out


def select(d, mask):
    """Sous-ensemble d'un dictionnaire renvoyé par matrix (masque booléen ou liste de codes de partie)."""
    if not (isinstance(mask, np.ndarray) and mask.dtype == bool):
        mask = np.isin(d["split"], np.atleast_1d(mask))
    n = len(d["stay_id"])
    return {k: (v[mask] if isinstance(v, np.ndarray) and v.ndim >= 1 and len(v) == n else v)
            for k, v in d.items()}
'''
(SRC_DIR / "tdtfl_data.py").write_text(TDTFL_MODULE.strip() + "\n", encoding="utf-8")
print("Écrit :", (SRC_DIR / "tdtfl_data.py").resolve())

Écrit : C:\Users\Stryve\Desktop\StryveSolution\Imagerie-Phd\P.H.D\New Impl\FUALA-w_28-06-2026\OP-FL-V2\src\tdtfl_data.py


## Étape 9 — Contrôles de cohérence

La cellule échoue si un contrôle ne passe pas : cohorte, découpages des trois régimes, variables, chargeur (normalisation calculée sur le seul train de la fédération, harmonisation), hash du protocole, et intégrité des fichiers d'OP-FL.

In [21]:
# ============================================================
# ÉTAPE 9 — CONTRÔLES DE COHÉRENCE
# ============================================================
sys.path.insert(0, str(SRC_DIR.resolve()))
import tdtfl_data
importlib.reload(tdtfl_data)

results = []
def check(name, ok):
    results.append((name, bool(ok)))
    print(("OK    " if ok else "ÉCHEC ") + name)

P = load_json(TDT_DIR / "protocol_tdtfl.json")
S = pd.read_parquet(TDT_DIR / "splits.parquet").set_index("patientunitstayid")
ST = pd.read_parquet(TDT_DIR / "stays.parquet").set_index("patientunitstayid")
H = pd.read_csv(TDT_DIR / "hospitals.csv", index_col="hospitalid")
S = S.loc[ST.index]
y = ST["y_mortality"].to_numpy()
hid = ST["hospitalid"].to_numpy()
rows_of = {int(h): np.flatnonzero(hid == h) for h in H.index}

# ---- cohorte ----
check("Un seul séjour par patient", ST["uniquepid"].is_unique)
check(f"Seuls des hôpitaux à >= {MIN_DEATHS} décès", (H["n_deaths"] >= MIN_DEATHS).all() and set(np.unique(hid)) == set(H.index))
check("Un phénotype valide pour chaque séjour", ST["phenotype"].isin(PHENOTYPES).all())
check("Strates et rôles cohérents", H["stratum"].isin(list(STRATA)).all() and H["role"].isin(["tune", "eval"]).all()
      and all(H.loc[H["stratum"] == s, "role"].eq("tune").sum() == round(TUNE_FRACTION * (H["stratum"] == s).sum()) for s in STRATA))

# ---- R1 ----
ok_codes = ok_prop = ok_deaths = ok_inner = True
for s in range(N_SEEDS):
    c = S[f"r1_s{s}"].to_numpy()
    inner = S[f"r1_inner_s{s}"].to_numpy()
    ok_codes &= set(np.unique(c)) == {0, 1, 2}
    frac = np.bincount(c, minlength=3) / len(c)
    ok_prop &= np.allclose(frac, [0.6, 0.2, 0.2], atol=0.01)
    for h, r in rows_of.items():
        k = y[r].sum()
        for code_, share in ((1, 0.2), (2, 0.2)):
            ok_deaths &= abs(y[r][c[r] == code_].sum() - share * k) <= 2
    ok_inner &= np.all((inner >= 0) == (c == 0)) and set(np.unique(inner[c == 0])) == set(range(N_INNER_FOLDS))
check("R1 : codes 0/1/2 présents pour chaque graine", ok_codes)
check("R1 : proportions globales 60/20/20 (à 1 point près)", ok_prop)
check("R1 : décès de D et du test à 2 près de 20 % dans chaque hôpital et chaque graine", ok_deaths)
check("R1 : plis internes définis sur le train seulement", ok_inner)
check("R1 : les graines donnent des découpages différents", (S["r1_s0"] != S["r1_s1"]).mean() > 0.2)

# ---- R2 ----
yr = ST["year"].to_numpy()
temporal_rows = ST["hospitalid"].map(H["temporal"].astype(bool)).to_numpy()
ok_r2 = True
for s in range(N_SEEDS):
    c = S[f"r2_s{s}"].to_numpy()
    ok_r2 &= np.all((c >= 0) == temporal_rows)
    ok_r2 &= np.isin(c[temporal_rows & (yr == YEARS[0])], [0, 1, 2]).all()
    ok_r2 &= np.isin(c[temporal_rows & (yr == YEARS[1])], [3, 4]).all()
check("R2 : hôpitaux temporels seulement, 2014 -> train14/D14/test14, 2015 -> D15/test15", ok_r2)

# ---- R3 ----
ok_pool = ok_rank = ok_budget = True
for s in range(N_SEEDS):
    c = S[f"r1_s{s}"].to_numpy()
    pool, rank = S[f"r3_pool_s{s}"].to_numpy(), S[f"r3_rank_s{s}"].to_numpy()
    ok_pool &= np.all((pool < 0) == (c == 2)) and np.all((rank < 0) == (c == 2))
    for h, r in rows_of.items():
        for p in (0, 1):
            rk = np.sort(rank[r][pool[r] == p])
            ok_rank &= np.array_equal(rk, np.arange(len(rk)))
            yy = y[r][pool[r] == p][np.argsort(rank[r][pool[r] == p])]
            for b in (50, 100):
                m = min(b // 2, len(yy))
                ok_budget &= abs(yy[:m].sum() - m * yy.mean()) <= 1.5
check("R3 : réservoirs et rangs définis exactement hors du test R1", ok_pool)
check("R3 : rangs d'acquisition = permutation 0..n-1 dans chaque réservoir", ok_rank)
check("R3 : décès d'un préfixe à 1,5 près de la proportion de l'hôpital", ok_budget)

# ---- variables ----
FR = pd.read_parquet(TDT_DIR / "features_raw.parquet").set_index("patientunitstayid")
M_ = pd.read_csv(TDT_DIR / "feature_meta.csv")
check("Variables : alignées sur les séjours, métadonnées cohérentes", set(FR.index) == set(ST.index)
      and FR.columns.tolist() == M_["name"].tolist())
check("Variables : aucune valeur infinie ; binaires dans {0, 1}", not np.isinf(FR.to_numpy()).any()
      and set(np.unique(FR[M_.loc[M_["kind"] == "bin", "name"]].to_numpy())) <= {0.0, 1.0})

# ---- chargeur ----
data = tdtfl_data.TDTData(TDT_DIR)
norm = data.normalizer("r1", 0)
j = data.targets("r1", role="eval")[0]
dj = data.matrix(j, "r1", 0, norm)
check("Chargeur : X fini, de la bonne forme", np.isfinite(dj["X"]).all() and dj["X"].shape == (len(rows_of[j]), len(M_)))
tr_rows = data.train_rows("r1", 0)
col = int(np.flatnonzero(data.cont)[0])
manual = np.nanmean(data.raw[S["r1_s0"].to_numpy() == 0][:, np.flatnonzero(data.cont)[0]])
check("Chargeur : normalisation calculée sur le train de la fédération uniquement",
      np.isclose(norm["mean"][0], manual, rtol=1e-4) and len(tr_rows) == int((S["r1_s0"] == 0).sum())
      and norm["n_rows"] == len(tr_rows))
fold = data.r3_fold(j)
fed = data.federation("r3", fold)
check("Chargeur : en R3, la fédération exclut le pli de la cible", j not in fed and
      all(data.r3_fold(h) != fold for h in fed) and data.normalizer("r3", 0, fold)["n_rows"] < norm["n_rows"])
pairs = [(i, jj) for i in data.federation() for jj in data.targets("r1", role=None) if i != jj and data.masked_variables(i, jj)]
if pairs:
    i, jj = pairs[0]
    v = data.masked_variables(i, jj)[0]
    d_ij = data.matrix(jj, "r1", 0, norm, also_mask=i)
    cols = M_.index[M_["var"] == v].to_numpy()
    obs_col = M_.index[(M_["var"] == v) & (M_["stat"] == "observed")][0]
    nobs_col = M_.index[(M_["var"] == v) & (M_["stat"] == "n_obs")][0]
    k_nobs = int(np.flatnonzero(np.flatnonzero(data.cont) == nobs_col)[0])
    expected_nobs = (0 - norm["mean"][k_nobs]) / norm["std"][k_nobs]
    cont_cols = [cc for cc in cols if M_.loc[cc, "stat"] in tdtfl_data.MISSING_STATS]
    ok_h = (np.all(d_ij["X"][:, cont_cols] == 0) and np.all(d_ij["X"][:, obs_col] == 0)
            and np.allclose(d_ij["X"][:, nobs_col], expected_nobs, atol=1e-5))
    check(f"Harmonisation : « {v} » (indisponible chez {i}) remise à manquant chez {jj}", ok_h)
else:
    print("(aucune paire avec variable masquée : contrôle d'harmonisation sans objet)")
own_ok = True
for h in data.federation():
    for v in data.unavailable(h)[:2]:
        c_obs = M_.index[(M_["var"] == v) & (M_["stat"] == "observed")][0]
        own_ok &= np.all(data.raw[data._rows[h], c_obs] == 0)
check("Harmonisation : variables indisponibles d'un hôpital masquées dans ses propres données", own_ok)
cal, dmask = data.r3_labeled(j, 0, 100)
check("Chargeur : budget R3 de 100 = 50 calibration + 50 D, disjoints et hors test",
      cal.sum() <= 50 and dmask.sum() <= 50 and not np.any(cal & dmask) and not np.any((cal | dmask) & (dj["split"] == 2)))

# ---- protocole et intégrité d'OP-FL ----
comp = {"hospitals": digest_df(H.reset_index()[["hospitalid"] + HOSP_COLS]),
        "phenotype_map": digest_df(pd.read_csv(TDT_DIR / "phenotype_map.csv")[["dx", "phenotype"]].sort_values("dx")),
        "splits": digest_df(S.reset_index().sort_values("patientunitstayid").reset_index(drop=True))}
check("Hash du protocole recalculé depuis les fichiers écrits", comp == P["components"])
opfl_now = hashlib.sha256(OPFL_PROTOCOL.read_bytes()).hexdigest() if OPFL_PROTOCOL.exists() else None
check("Protocole OP-FL (splits/protocol.json) inchangé", opfl_now == OPFL_PROTOCOL_SHA)

failed = [n for n, ok in results if not ok]
if failed:
    raise AssertionError(f"{len(failed)} contrôle(s) en échec : {failed}")
print(f"\nTous les contrôles passent ({len(results)}).")

OK    Un seul séjour par patient
OK    Seuls des hôpitaux à >= 20 décès
OK    Un phénotype valide pour chaque séjour
OK    Strates et rôles cohérents
OK    R1 : codes 0/1/2 présents pour chaque graine
OK    R1 : proportions globales 60/20/20 (à 1 point près)
OK    R1 : décès de D et du test à 2 près de 20 % dans chaque hôpital et chaque graine
OK    R1 : plis internes définis sur le train seulement
OK    R1 : les graines donnent des découpages différents
OK    R2 : hôpitaux temporels seulement, 2014 -> train14/D14/test14, 2015 -> D15/test15
OK    R3 : réservoirs et rangs définis exactement hors du test R1
OK    R3 : rangs d'acquisition = permutation 0..n-1 dans chaque réservoir
OK    R3 : décès d'un préfixe à 1,5 près de la proportion de l'hôpital
OK    Variables : alignées sur les séjours, métadonnées cohérentes
OK    Variables : aucune valeur infinie ; binaires dans {0, 1}
OK    Chargeur : X fini, de la bonne forme
OK    Chargeur : normalisation calculée sur le train de la fédération

## Étape 10 — Effectifs utiles aux expériences et fiche de données

Ces tableaux fixent ce que les expériences peuvent raisonnablement mesurer, avant de les lancer :
* **décès dans D par hôpital** : la puissance de la porte en dépend directement ;
* **faisabilité de E2** : nombre de couples (hôpital, phénotype) qui ont au moins 5 ou 10 décès dans D ;
* **régime temporel**, **budgets R3** et **tâche AKI** (secondaire).

In [22]:
# ============================================================
# ÉTAPE 10 — EFFECTIFS UTILES AUX EXPÉRIENCES + FICHE DE DONNÉES
# ============================================================
ST = pd.read_parquet(TDT_DIR / "stays.parquet").set_index("patientunitstayid")
S = pd.read_parquet(TDT_DIR / "splits.parquet").set_index("patientunitstayid").loc[ST.index]
H = pd.read_csv(TDT_DIR / "hospitals.csv", index_col="hospitalid")
P = load_json(TDT_DIR / "protocol_tdtfl.json")
order = list(STRATA)

# 1) décès dans D et dans le test, par hôpital (moyenne sur les graines)
dd = np.mean([ST.loc[S[f"r1_s{s}"] == 1].groupby("hospitalid")["y_mortality"].sum() for s in range(N_SEEDS)], axis=0)
dt = np.mean([ST.loc[S[f"r1_s{s}"] == 2].groupby("hospitalid")["y_mortality"].sum() for s in range(N_SEEDS)], axis=0)
hd = pd.DataFrame({"deces_D": dd, "deces_test": dt}, index=np.sort(ST["hospitalid"].unique())).join(H[["stratum", "role"]])
tab_d = hd.groupby("stratum").agg(hopitaux=("deces_D", "size"), deces_D_moyen=("deces_D", "mean"),
                                  deces_D_min=("deces_D", "min"), deces_D_max=("deces_D", "max"),
                                  deces_test_moyen=("deces_test", "mean")).reindex(order).round(1)
print("R1 — décès par hôpital dans D et dans le test (moyenne sur les graines) :")
display(tab_d)

# 2) faisabilité de E2 : couples (hôpital d'évaluation, phénotype)
cells = np.mean([ST.loc[S[f"r1_s{s}"] == 1].groupby(["hospitalid", "phenotype"])["y_mortality"].sum()
                 .unstack(fill_value=0).reindex(columns=PHENOTYPES, fill_value=0) for s in range(N_SEEDS)], axis=0)
cells = pd.DataFrame(cells, index=np.sort(ST["hospitalid"].unique()), columns=PHENOTYPES)
ev = cells.loc[H.index[H["role"] == "eval"]]
rows_e2 = []
for s_name in order:
    sub = ev.loc[H.loc[ev.index, "stratum"] == s_name]
    rec = {"strate": s_name, "couples": sub.size}
    for k in SUBGROUP_DEATH_THRESHOLDS:
        rec[f">= {k} décès dans D"] = f"{(sub >= k).to_numpy().mean():.0%}"
    rows_e2.append(rec)
tab_e2 = pd.DataFrame(rows_e2).set_index("strate")
print("E2 — part des couples (hôpital d'évaluation, phénotype) avec assez de décès dans D :")
display(tab_e2)
print("Décès moyens dans D par phénotype et par strate (hôpitaux d'évaluation) :")
display(ev.groupby(H.loc[ev.index, "stratum"]).mean().reindex(order).round(1))

# 3) régime temporel
inv2 = {v: k for k, v in R2_CODES.items()}
t0 = ST.assign(part=S["r2_s0"].map(inv2)).dropna(subset=["part"])
tab_t = (t0.groupby(["hospitalid", "part"])["y_mortality"].sum().unstack("part").join(H["stratum"])
         .groupby("stratum").mean().reindex(order).round(1)[list(R2_CODES)])
tab_t.insert(0, "hopitaux", H[H["temporal"].astype(bool)].groupby("stratum").size().reindex(order))
print("R2 — décès moyens par hôpital et par partie (graine 0) :")
display(tab_t)

# 4) budgets R3
pool0 = S["r3_pool_s0"].to_numpy()
hid = ST["hospitalid"].to_numpy()
res_min = pd.Series({h: min((pool0[hid == h] == 0).sum(), (pool0[hid == h] == 1).sum()) for h in H.index})
tab_r3 = pd.DataFrame({f"b = {b}": [f"{(res_min[H.index[H['stratum'] == s]] >= b - b // 2).mean():.0%}" for s in order]
                       for b in R3_BUDGETS}, index=order)
tab_r3["décès attendus dans D (b = 100)"] = [round(50 * H.loc[H["stratum"] == s, "mortality"].mean(), 1) for s in order]
tab_r3.index.name = "strate"
print("R3 — part des hôpitaux qui peuvent fournir chaque budget :")
display(tab_r3)

# 5) tâche secondaire AKI
if "y_aki" in ST.columns:
    ak = ST[ST["aki_ascertained"].astype(bool)].groupby("hospitalid").agg(evaluables=("y_aki", "size"), aki=("y_aki", "sum"))
    ak = ak.join(H["stratum"])
    tab_aki = ak.groupby("stratum").agg(evaluables=("evaluables", "sum"), aki=("aki", "sum"),
                                        hopitaux_20_aki=("aki", lambda a: int((a >= 20).sum()))).reindex(order)
    print(f"AKI (tâche secondaire) : {int(ak['aki'].sum())} événements chez {int(ak['evaluables'].sum())} séjours évaluables ; "
          f"{int((ak['aki'] >= 20).sum())} hôpitaux avec >= 20 événements")
    display(tab_aki)

# 6) phénotypes et disponibilité
tab_ph = ST.groupby("phenotype").agg(sejours=("y_mortality", "size"), deces=("y_mortality", "sum"),
                                     mortalite=("y_mortality", "mean"), operatoire=("operative", "mean")).reindex(PHENOTYPES)
tab_ph["mortalite"] = tab_ph["mortalite"].round(3)
tab_ph["operatoire"] = tab_ph["operatoire"].round(3)
AV = pd.read_csv(TDT_DIR / "availability.csv", index_col="hospitalid")
tab_av = pd.DataFrame({"hopitaux_indisponibles": (AV == 0).sum()}).query("hopitaux_indisponibles > 0")

for name, df in {"tdtfl_deces_par_hopital": hd, "tdtfl_e2_cellules_D": cells, "tdtfl_strates_D": tab_d,
                 "tdtfl_e2_faisabilite": tab_e2, "tdtfl_temporel": tab_t, "tdtfl_r3_budgets": tab_r3,
                 "tdtfl_phenotypes": tab_ph, "tdtfl_disponibilite": tab_av}.items():
    df.to_csv(REPORTS / f"{name}.csv")

# ---- fiche de données ----
el_sum = H.groupby("stratum").agg(hopitaux=("n_stays", "size"), sejours=("n_stays", "sum"), deces=("n_deaths", "sum"),
                                  reglage=("role", lambda r: int((r == "tune").sum())),
                                  evaluation=("role", lambda r: int((r == "eval").sum())),
                                  temporel=("temporal", lambda t: int(t.astype(bool).sum()))).reindex(order)
card = f"""# Fiche de données — TDT-FL, protocole {P['version']} (eICU-CRD)

Générée le {datetime.now():%Y-%m-%d %H:%M} par `notebooks/02_protocole_tdtfl.ipynb`. Hash du protocole : `{P['hash']}`.
Source : sorties du notebook 01 d'OP-FL (`{OPFL_DIR}`), lues sans modification.

## Hôpitaux
Éligibles : au moins {MIN_DEATHS} décès sur {YEARS[0]}-{YEARS[1]} ; strates par nombre de décès {STRATA}.
Réglage : {TUNE_FRACTION:.0%} de chaque strate (cibles de réglage des hyperparamètres, sources pour tous). Évaluation : les autres.

{md_table(el_sum)}

Séjours retenus : {len(ST)} ; décès : {int(ST['y_mortality'].sum())} ; mortalité : {ST['y_mortality'].mean():.2%}.

## Phénotypes (`apacheadmissiondx`, texte entre parenthèses retiré, priorité dans l'ordre)
{md_table(tab_ph)}

Axe secondaire : opératoire si `unitadmitsource` ∈ {sorted(OPERATIVE_SOURCES)}. Correspondance complète : `phenotype_map.csv`.

## Découpages ({N_SEEDS} graines, stratifiés sur décès × phénotype, allocation par blocs)
- R1 statique : train / D / test = 60 / 20 / 20 dans chaque hôpital ; {N_INNER_FOLDS} plis internes du train.
- R2 temporel : hôpitaux à ≥ {MIN_DEATHS_PER_YEAR} décès par an ; {YEARS[0]} -> train14 / D14 / test14 (60/20/20), {YEARS[1]} -> D15 / test15 (50/50).
- R3 hôpital non vu : test = test R1 ; budgets {list(R3_BUDGETS)} pris pour moitié dans un réservoir de calibration et pour moitié dans un réservoir D, dans un ordre d'acquisition stratifié ; fédération = hôpitaux hors du pli R3 de la cible ({N_R3_FOLDS} plis).
- R4 dérive simulée : lots tirés des réservoirs D et test de R1 (voir la spécification).

### Décès par hôpital dans D et le test (R1, moyenne sur les graines)
{md_table(tab_d)}

### Faisabilité de E2 (couples hôpital d'évaluation × phénotype)
{md_table(tab_e2)}

### Régime temporel (graine 0)
{md_table(tab_t)}

### Budgets R3
{md_table(tab_r3)}

## Variables ({len(M_)} colonnes : {int((M_['kind'] == 'cont').sum())} continues, {int((M_['kind'] == 'bin').sum())} binaires)
- {len(VARS)} variables temporelles d'OP-FL (log1p pour les variables asymétriques) : moyenne, minimum, maximum, dernière valeur, nombre d'observations, observée ou non ; diurèse totale.
- Statiques : âge, taille, poids, sexe, indicateurs de valeur manquante. One-hot : {N_TOP_DX} diagnostics + autre + manquant (vocabulaire des hôpitaux de réglage), phénotype, type d'unité, provenance.
- Normalisation globale au chargement (`src/tdtfl_data.py`), sur le split train de la fédération du régime ; valeur manquante = 0 après z-score.
- Disponibilité : variable observée chez ≥ {AVAIL_MIN_COVERAGE:.0%} des séjours de l'hôpital. Une variable indisponible chez la source est remise à manquant chez la cible avant d'appliquer le modèle de la source.

{md_table(tab_av) if len(tab_av) else "Aucune variable indisponible."}

## Usage des parties
- train : entraînement du modèle local ; recalibration de Platt des partenaires sur les patients de la cible.
- D : décisions uniquement (choix du partenaire, porte).
- test : mesure uniquement.
"""
(TDT_DIR / "DATA_CARD_TDTFL.md").write_text(card, encoding="utf-8")
print("Fichiers :", sorted(p.name for p in TDT_DIR.iterdir() if p.is_file()))
print("Rapports :", sorted(p.name for p in REPORTS.iterdir()))

R1 — décès par hôpital dans D et dans le test (moyenne sur les graines) :


,hopitaux,deces_D_moyen,deces_D_min,deces_D_max,deces_test_moyen
stratum,,,,,
petit,50,6.8,3.7,11.6,6.8
moyen,40,17.0,11.8,30.1,17.0
grand,11,45.8,30.2,65.0,45.6


E2 — part des couples (hôpital d'évaluation, phénotype) avec assez de décès dans D :


,couples,>= 5 décès dans D,>= 10 décès dans D
strate,,,
petit,200,1%,0%
moyen,160,21%,0%
grand,45,87%,31%


Décès moyens dans D par phénotype et par strate (hôpitaux d'évaluation) :


,sepsis,cardiovasculaire,neurologique,respiratoire,autre
stratum,,,,,
petit,1.7,2.0,0.8,1.2,1.0
moyen,4.0,4.6,2.7,3.1,2.4
grand,8.5,12.3,9.2,7.1,6.3


... (3 outputs left out)

,b = 50,b = 100,b = 200,décès attendus dans D (b = 100)
strate,,,,
petit,100%,100%,88%,4.6
moyen,100%,100%,100%,5.4
grand,100%,100%,100%,5.3


AKI (tâche secondaire) : 2382 événements chez 64949 séjours évaluables ; 50 hôpitaux avec >= 20 événements


,evaluables,aki,hopitaux_20_aki
stratum,,,
petit,17233,639.0,10
moyen,28176,1040.0,29
grand,19540,703.0,11


Fichiers : ['DATA_CARD_TDTFL.md', 'availability.csv', 'coverage.csv', 'feature_info.json', 'feature_meta.csv', 'features_raw.parquet', 'hospitals.csv', 'phenotype_map.csv', 'protocol_tdtfl.json', 'splits.parquet', 'stays.parquet']
Rapports : ['tdtfl_deces_par_hopital.csv', 'tdtfl_disponibilite.csv', 'tdtfl_e2_cellules_D.csv', 'tdtfl_e2_faisabilite.csv', 'tdtfl_phenotypes.csv', 'tdtfl_r3_budgets.csv', 'tdtfl_strates_D.csv', 'tdtfl_temporel.csv']


## Et ensuite

1. **Relisez `phenotype_map.csv`** (surtout les diagnostics classés « autre ») et corrigez si besoin par `PHENOTYPE_OVERRIDES`, **avant** E1. Une correction change le hash : supprimez alors `protocol_tdtfl.json` volontairement et ré-exécutez.
2. **Fixez avec votre directeur les seuils go/no-go** (G1 à G6) et le minimum de décès par sous-groupe pour E2, à la lumière des tableaux de l'étape 10, puis consignez-les.
3. Notebook suivant, `03_modeles_locaux_E0` : un modèle par hôpital (régression logistique L2, régularisation choisie par les plis internes), modèle global (FedAvg) et centralisé, résultats E0 par strate.